# Варіант 2. Автоматична класифікація службових повідомлень
**Тема 4, заняття 15 · Індивідуальний проєкт · 40 хв.**

**Мета:** навчити класифікатор визначати, до якої з чотирьох категорій належить нове повідомлення.

Категорії: **Забезпечення, Технічна підтримка, Звітність, Навчання.**

**Методи:** TF–IDF → Naive Bayes → accuracy, F1, матриця помилок.

**Результат:** звіт про точність і прогноз для власного нового повідомлення.

**План (40 хв):** 5 — інструктаж; 10 — дані й векторизація; 15 — навчання/оцінювання; 10 — власні тексти та висновок.

Усі повідомлення синтетичні.

In [1]:
from pathlib import Path
import csv
from collections import Counter

DATA = Path("Варіант_2_класифікація_48.csv")
if not DATA.exists():
    DATA = Path("../Варіант_2_класифікація_48.csv")
with DATA.open(encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))
texts = [r["текст"] for r in rows]
labels = [r["категорія"] for r in rows]
print("Кількість повідомлень:", len(rows))
print("Категорії:", dict(Counter(labels)))

Кількість повідомлень: 48
Категорії: {'Забезпечення': 12, 'Технічна підтримка': 12, 'Звітність': 12, 'Навчання': 12}


## 1. Поділ на навчальні та тестові приклади
Тестові повідомлення **не повинні використовуватися для навчання**. `stratify` зберігає представництво чотирьох категорій.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.25, random_state=42, stratify=labels
)
model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("clf", MultinomialNB()),
])
model.fit(X_train, y_train)
print(f"Навчальних: {len(X_train)}; тестових: {len(X_test)}")

Навчальних: 36; тестових: 12


## 2. Вимірювання точності
- **Accuracy** — частка всіх правильних класифікацій.
- **Precision** — наскільки влучні призначення певної категорії.
- **Recall** — яку частку повідомлень категорії знайдено.
- **F1** — баланс точності й повноти.

Невеликий синтетичний тест не доводить придатності до використання у реальній системі.

In [3]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

pred = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print(classification_report(y_test, pred, zero_division=0))
categories = sorted(set(labels))
print("Матриця помилок (рядки — істина, стовпці — прогноз):")
print("Категорії:", categories)
print(confusion_matrix(y_test, pred, labels=categories))

Accuracy: 0.667
                    precision    recall  f1-score   support

      Забезпечення       1.00      0.67      0.80         3
         Звітність       0.50      0.67      0.57         3
          Навчання       1.00      0.33      0.50         3
Технічна підтримка       0.60      1.00      0.75         3

          accuracy                           0.67        12
         macro avg       0.78      0.67      0.66        12
      weighted avg       0.78      0.67      0.66        12

Матриця помилок (рядки — істина, стовпці — прогноз):
Категорії: ['Забезпечення', 'Звітність', 'Навчання', 'Технічна підтримка']
[[2 1 0 0]
 [0 2 0 1]
 [0 1 1 1]
 [0 0 0 3]]


## 3. Випробування нових текстів
Змініть або додайте власні **дві навчальні службові фрази**. Подивіться, чи обирає модель правильну категорію.

In [4]:
# Два власні навчальні службові тексти
new_messages = [
    "До кафедри потрібно доставити додаткові мережеві кабелі.",
    "Під час входу в систему з’являється повідомлення про помилку.",
    "Слухачі опрацювали лекцію про нейронні мережі та виконали практичну вправу.",
    "Прошу надати зведені дані про успішність групи за семестр.",
]
for msg, category in zip(new_messages, model.predict(new_messages)):
    print(f"{category:21} | {msg}")

Забезпечення          | До кафедри потрібно доставити додаткові мережеві кабелі.
Технічна підтримка    | Під час входу в систему з’являється повідомлення про помилку.
Навчання              | Слухачі опрацювали лекцію про нейронні мережі та виконали практичну вправу.
Звітність             | Прошу надати зведені дані про успішність групи за семестр.


### Аналіз помилок на тестовій вибірці

In [5]:
print("Неправильно класифіковані тестові повідомлення:")
for text, true, p in zip(X_test, y_test, pred):
    if true != p:
        print(f"істина: {true:18} | прогноз: {p:18} | {text}")

Неправильно класифіковані тестові повідомлення:
істина: Забезпечення       | прогноз: Звітність          | Підготовлено накладну на видачу обладнання зі складу.
істина: Звітність          | прогноз: Технічна підтримка | Звітність оновлено після перевірки первинних записів.
істина: Навчання           | прогноз: Технічна підтримка | На семінарі обговорено правила складання промптів.
істина: Навчання           | прогноз: Звітність          | Підготовлено навчальний ноутбук із прикладами токенізації.


## 4. Звіт слухача

**Кількість повідомлень у навчанні / тесті:** 36 / 12 (по 9 і 3 повідомлення кожної з 4 категорій завдяки `stratify`).

**Accuracy:** 0.667 (8 з 12 тестових повідомлень). Macro F1 = 0.66: Забезпечення — 0.80, Технічна підтримка — 0.75, Звітність — 0.57, Навчання — 0.50.

**Категорії, які модель плутає:** за матрицею помилок:
- *Навчання* — найгірше (recall 0.33): «правила складання промптів» → Технічна підтримка, «навчальний ноутбук із прикладами токенізації» → Звітність (слово «підготовлено» часто трапляється у звітах).
- *Звітність → Технічна підтримка*: «Звітність оновлено після перевірки первинних записів» (слово «оновлено» типове для техпідтримки).
- *Забезпечення → Звітність*: «Підготовлено накладну на видачу обладнання» — знову вплив слова «підготовлено».
- Технічна підтримка має recall 1.00, але precision 0.60 — модель «перетягує» до неї чужі повідомлення.

**Два власні тексти і прогноз:**
- «Слухачі опрацювали лекцію про нейронні мережі та виконали практичну вправу.» → **Навчання** ✔ (правильно)
- «Прошу надати зведені дані про успішність групи за семестр.» → **Звітність** ✔ (правильно)
- Приклади з ноутбука (мережеві кабелі → Забезпечення; помилка входу → Технічна підтримка) також класифіковано правильно.

**Чому висока точність на маленькій синтетичній вибірці не гарантує успіху на реальних рапортах?**
1. Тест містить лише 12 повідомлень (по 3 на клас): одна помилка змінює accuracy на ~8 %, тож оцінка дуже нестабільна й залежить від `random_state`.
2. Синтетичні тексти короткі, однотипні та без помилок; реальні рапорти довші, містять скорочення, опечатки, змішані теми.
3. Модель спирається на окремі слова («підготовлено», «оновлено»), а не на зміст — у реальних текстах ці слова розподілені інакше.
4. У реальності класи незбалансовані й з'являються нові категорії, яких модель не бачила.

**Обмеження методу:** TF–IDF + Naive Bayes — модель «мішка слів»: не враховує синоніми, порядок слів і контекст, а словоформи української мови розглядає як різні ознаки.

**Висновок:** класифікатор TF–IDF + MultinomialNB досяг accuracy 0.667 на 12 тестових повідомленнях; найкраще розпізнає Забезпечення та Технічну підтримку, найгірше — Навчання. Для поліпшення варто збільшити набір даних, застосувати крос-валідацію (наприклад, `StratifiedKFold` із 5 фолдами) замість одного поділу, лематизацію або символьні n-грами, і порівняти з іншими моделями (LogisticRegression, LinearSVC).